In [2]:
from pathlib import Path
import json
import os

# Simple image indexer: finds images under 'Images/' and groups by filename.
# Stores relative paths (relative to project root) for `path` and `other_paths`.
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp'}

def build_index_simple(images_root='Images', project_root=None):
    root = Path(images_root)
    project_root = Path(project_root or '.').resolve()
    images = {}

    for idx, p in enumerate(root.rglob('*')):

        # ### IDX LIMIT
        # if idx >= 10:
        #     break

        if not p.is_file():
            print(f"Skipping non-file: {p}")
            continue
        if p.suffix.lower() not in IMAGE_EXTS:
            continue

        name = p.name
        abs_path = p.resolve()
        try:
            rel_path = str(abs_path.relative_to(project_root))
        except ValueError:
            # file is outside project_root -> use os.path.relpath (may include ..) or absolute
            rel_path = os.path.relpath(str(abs_path), str(project_root))
            print(f"Warning: file {abs_path} is outside project root {project_root}, using relative path {rel_path}")
        
        rel_path = rel_path.replace('\\', '/')  # Replace backslashes with forward slashes
        
        label = p.parent.name

        if name not in images:
            images[name] = {
                'image_name': name,
                'path': rel_path,
                'other_paths': [],
                'labels': [label] if label else [],
            }
        else:
            existing_paths = images[name]['other_paths'] + [images[name]['path']]
            print(f"Found duplicate image name: {name}, adding additional path.")
            if rel_path not in existing_paths:
                images[name]['other_paths'].append(rel_path)
            if label and label not in images[name]['labels']:
                images[name]['labels'].append(label)

    return list(images.values())

# Example: idx = build_index_simple('Images')
# To write to disk: json.dump(idx, open('image_index.json','w'), indent=2)


In [12]:
# Run the simple indexer and write output to image_index.json
testset_name = 'random'

# Get the actual project root - go up from testsets folder
repo_root = Path('.').resolve().parent if Path('.').resolve().name == 'testsets' else Path('.').resolve()

# Build the image index
images_dir = repo_root / 'Images' / f'{testset_name} Testset'

# DEBUG: Check what path we're searching
print(f"Looking for images in: {images_dir}")
print(f"Directory exists: {images_dir.exists()}")
if images_dir.exists():
    print(f"Contents: {list(images_dir.iterdir())[:10]}")  # Show first 10 items
else:
    print(f"Parent directory exists: {images_dir.parent.exists()}")
    if images_dir.parent.exists():
        print(f"Available testsets: {list(images_dir.parent.iterdir())}")

index = build_index_simple(str(images_dir), project_root=repo_root)

# Write to output file
output_dir = repo_root / 'testsets'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{testset_name}_testset.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(index, f, ensure_ascii=False, indent=2)

print(f"Wrote index with {len(index)} unique image names to {output_path}.")

Looking for images in: /app/Images/random Testset
Directory exists: True
Contents: [PosixPath('/app/Images/random Testset/Donald Trump'), PosixPath('/app/Images/random Testset/Giorgia Meloni'), PosixPath('/app/Images/random Testset/Hugh Jackman'), PosixPath('/app/Images/random Testset/Lionel Messi'), PosixPath('/app/Images/random Testset/None')]
Skipping non-file: /app/Images/random Testset/Donald Trump
Skipping non-file: /app/Images/random Testset/Giorgia Meloni
Skipping non-file: /app/Images/random Testset/Hugh Jackman
Skipping non-file: /app/Images/random Testset/Lionel Messi
Skipping non-file: /app/Images/random Testset/None
Found duplicate image name: LP_23299305.jpg, adding additional path.
Wrote index with 54 unique image names to /app/testsets/random_testset.json.
